# PSDCodec Deployment Animation Demo

This notebook is a thin deployment walkthrough built around the exported `models/exports/demo` artifacts. It evaluates a batch of raw campaign frames, selects representative examples across the distortion range, and renders a `matplotlib.animation.FuncAnimation` that cycles through many deployment examples.


## Notebook Flow

1. Load the exported deployment artifacts and restore the ONNX encoder + PyTorch decoder boundary.
2. Evaluate a configurable batch of campaign frames through the deployed codec.
3. Summarize deployment readiness and select a representative animation subset.
4. Render a `FuncAnimation` showing original, preprocessing-only, and codec PSDs for many examples.


In [58]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
src_root = project_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

# Reload notebook-facing modules so rerunning this cell picks up local code edits.
for module_name in ("interfaces.demo_animation", "interfaces.deployment"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

demo_animation = importlib.import_module("interfaces.demo_animation")
deployment = importlib.import_module("interfaces.deployment")
build_animation_frame_summary_rows = demo_animation.build_animation_frame_summary_rows
create_deployment_animation = demo_animation.create_deployment_animation
select_animation_frame_reports = demo_animation.select_animation_frame_reports
build_deployment_demo_summary_rows = deployment.build_deployment_demo_summary_rows
create_deployment_service = deployment.create_deployment_service
evaluate_deployment_batch = deployment.evaluate_deployment_batch

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FCFCFD",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
    }
)

In [59]:
export_dir = project_root / "models" / "exports" / "demo"
summary_path = export_dir / "training_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(
        "Expected a completed demo export at models/exports/demo. Finish the canonical "
        "training run before opening this notebook."
    )

training_summary = json.loads(summary_path.read_text(encoding="utf-8"))
service, artifacts = create_deployment_service(export_dir)

best_epoch_index = training_summary["best_epoch_index"]
best_score = training_summary["best_selection_score"]
display(
    Markdown(
        f"**Loaded export:** `{export_dir}`\n\n"
        f"**Best checkpoint epoch:** {best_epoch_index + 1}\n\n"
        f"**Best deployment score:** {best_score:.6f}"
    )
)
artifacts.checkpoint_path

**Loaded export:** `/home/marti/Code/PSDCodec/models/exports/demo`

**Best checkpoint epoch:** 77

**Best deployment score:** 0.828894

PosixPath('/home/marti/Code/PSDCodec/models/checkpoints/demo/best.pt')

In [60]:
batch_frame_count = 3
animation_frame_count = 24
plot_dbm = True
campaign_include_globs = ["test_full_tdt"]
campaign_exclude_globs: list[str] = []
node_include_globs = ["Node3.csv"]
frame_indices: list[int] | None = None

display(
    Markdown(
        "**Frame-selection boundary:**\n\n"
        f"- Campaign include globs: `{campaign_include_globs}`\n"
        f"- Campaign exclude globs: `{campaign_exclude_globs}`\n"
        f"- Node include globs: `{node_include_globs}`\n"
        f"- Explicit frame indices: `{frame_indices}`\n"
        f"- PSD plotting domain: `{'dBm' if plot_dbm else 'linear power [mW]'}`"
    )
)

batch_report = evaluate_deployment_batch(
    service,
    artifacts,
    max_frames=batch_frame_count,
    frame_indices=frame_indices,
    campaign_include_globs=campaign_include_globs,
    campaign_exclude_globs=campaign_exclude_globs,
    node_include_globs=node_include_globs,
    task_config=artifacts.experiment_config.task,
)

display(
    Markdown(
        "**Analytical demo summary:** codec dimensions and observed bitrate density "
        "for the exact export and evaluated batch."
    )
)
analysis_frame = pd.DataFrame(
    build_deployment_demo_summary_rows(
        artifacts,
        batch_report=batch_report,
    )
)
display(analysis_frame)

summary_frame = pd.DataFrame(
    [
        {"metric": key, "value": value}
        for key, value in batch_report.summary.to_display_dict().items()
    ]
)
display(summary_frame)

display(
    Markdown(
        f"**Readiness verdict:** `{batch_report.assessment.verdict}`\n\n"
        + "\n".join(f"- {reason}" for reason in batch_report.assessment.reasons)
    )
)

**Frame-selection boundary:**

- Campaign include globs: `['test_full_tdt']`
- Campaign exclude globs: `[]`
- Node include globs: `['Node3.csv']`
- Explicit frame indices: `None`
- PSD plotting domain: `dBm`

**Analytical demo summary:** codec dimensions and observed bitrate density for the exact export and evaluated batch.

,section,metric,value,units
0,Spectrum,Original PSD bins N,4096,bins
1,Spectrum,Preprocessed bins N_r,1024,bins
2,Spectrum,Deterministic reduction N / N_r,4.0,x
3,Latent,Latent positions M,512,vectors
4,Latent,Embedding dimension d,8,scalars/vector
5,Latent,Latent tensor shape,512 x 8,M x d
6,Latent,Continuous latent scalar count M*d,4096,scalars
7,VQ,Codebook size J,512,codewords
8,VQ,Runtime codebook shape,512 x 8,J x d
9,VQ,Index symbols per frame,512,symbols


,metric,value
0,frame_count,3
1,all_roundtrip_equal,True
2,packet_bits_mean,4517.666667
3,packet_bits_std,6.182412
4,packet_bits_min,4509
5,packet_bits_max,4523
6,rate_proxy_bits_mean,4524.174165
7,rate_proxy_bits_std,6.522358
8,psd_distortion_mean,0.01455
9,psd_distortion_std,0.000428


**Readiness verdict:** `borderline`

- Packet decode matches the encode-time reconstruction on every frame.
- Mean PSD distortion is low enough for a promising deployment candidate (0.015).
- Dominant-peak amplitude preservation looks acceptable (0.25 dB mean error).
- Mean dominant-peak frequency error is 250.7 kHz.
- Packet size is operationally stable (0.1% relative std).

In [61]:
animation_reports = select_animation_frame_reports(
    batch_report,
    frame_count=animation_frame_count,
)

selection_frame = pd.DataFrame(build_animation_frame_summary_rows(animation_reports))
selection_frame

,frame_index,sequence_id,timestamp_utc,packet_bits,psd_distortion,peak_frequency_error_khz,peak_power_error_db
0,1,test_full_tdt/Node3,2026-03-19 22:12:04,4521,0.014158,15.869141,0.297831
1,2,test_full_tdt/Node3,2026-03-19 22:14:04,4509,0.014346,726.806641,0.240820
2,0,test_full_tdt/Node3,2026-03-19 22:10:04,4523,0.015145,9.521484,0.221832


## Animated Deployment Walkthrough

The animation below cycles through representative deployment examples across the evaluated distortion range. The top panel shows the current PSD reconstruction, the bottom-left panel places the current frame in the packet-size / distortion distribution, and the bottom-right panel prints the frame-local deployment metrics.


In [62]:
interval_ms = 900
animation = create_deployment_animation(
    animation_reports,
    interval_ms=interval_ms,
    show_noise_floor=False,
    plot_dbm=plot_dbm,
)
animation_html = HTML(animation.to_jshtml())
plt.close(animation._fig)
animation_html

## Reading The Animation

- `Original PSD` is the reference campaign frame in linear power.
- `Preprocessing-only` shows the deterministic baseline before the learned codec stage.
- `Codec reconstruction` is the full deployed round-trip through the ONNX encoder and server decoder.
- The highlighted point in the lower-left panel shows where the current frame sits in the evaluated packet-size / distortion trade-off.
- The text panel exposes the exact per-frame packet budget and reconstruction metrics so you can correlate visible spectral errors with deployment costs.
